# Guide 3 — Grid Mapping (turning the board into a checkerboard)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

The camera sees the board from an angle, maybe tilted or even upside down. But we
want to talk about neat squares like `A1`, `B3`, `C5`. **Grid mapping** is the math
that lines up the tilted camera picture with a perfect checkerboard.

The trick that makes it work: the four corner markers are labeled by their *job*
(top-left, top-right, …), not by where they show up in the picture. So even if the
camera is upside down, `A1` is still `A1`.

This is all real code. You mostly run it and read the notes.


### How this guide fits in

**Depends on:** Guide 1 (grid size, `BOARD_CORNERS`). **Used by:** Guides 4, 5.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Naming squares

`pos_name` turns a row and column number into a name like `B3`. `parse_pos` does
the reverse. Simple but used everywhere.


In [ ]:
def pos_name(row, col):
    return f'{chr(ord("A") + row)}{col + 1}'


def parse_pos(pos):
    row = ord(pos[0].upper()) - ord('A')
    col = int(pos[1:]) - 1
    return row, col

### Lining up the board

These functions do the actual "un-tilting" using a math tool called a *perspective
transform*. `board_source_corners` finds the board's corners in the camera picture,
and the transform functions map between the camera view and the neat grid.

The key idea is in `board_grid_corners`: because the corners are always listed in
the same order (top-left, top-right, bottom-right, bottom-left), the math fixes any
rotation automatically.


In [ ]:
def board_source_corners(frame_shape):
    height, width = frame_shape[:2]
    if BOARD_CORNERS is not None:
        return np.float32(BOARD_CORNERS)
    return np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])


def board_grid_corners():
    """Canonical logical orientation: A1 is always the top-left cell.

    board_source_corners() is ordered by marker role (TL, TR, BR, BL), not by
    where each marker happens to appear in the camera image. The homography
    therefore corrects 90/180-degree camera rotation automatically.
    """
    return np.float32([
        [0, 0], [GRID_COLS, 0], [GRID_COLS, GRID_ROWS], [0, GRID_ROWS],
    ])


def board_transform(frame_shape):
    return cv2.getPerspectiveTransform(board_source_corners(frame_shape), board_grid_corners())


def frame_transform(frame_shape):
    return cv2.getPerspectiveTransform(board_grid_corners(), board_source_corners(frame_shape))

### Cutting out one square

`crop_grid_cell` cuts out just one square of the board and flattens it into a neat
picture we can hand to YOLO. The other helpers here turn the un-tilted view back
into camera coordinates and draw the grid lines you see on screen.


In [ ]:
def warp_board_to_canonical(frame, cell_pixels=120):
    """Perspective-warp the full board so semantic TL marker is image TL."""
    width = int(GRID_COLS * cell_pixels)
    height = int(GRID_ROWS * cell_pixels)
    destination = np.float32([
        [0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1],
    ])
    transform = cv2.getPerspectiveTransform(board_source_corners(frame.shape), destination)
    return cv2.warpPerspective(frame, transform, (width, height))


def draw_canonical_board(frame, selected_pos=None, cell_pixels=120):
    """Return an oriented board preview with A1 visibly in the top-left."""
    board = warp_board_to_canonical(frame, cell_pixels=cell_pixels)
    height, width = board.shape[:2]
    for col in range(GRID_COLS + 1):
        x = int(round(col * width / GRID_COLS))
        cv2.line(board, (min(x, width - 1), 0), (min(x, width - 1), height - 1), (0, 255, 0), 2)
    for row in range(GRID_ROWS + 1):
        y = int(round(row * height / GRID_ROWS))
        cv2.line(board, (0, min(y, height - 1)), (width - 1, min(y, height - 1)), (0, 255, 0), 2)
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            x = int(col * width / GRID_COLS) + 8
            y = int(row * height / GRID_ROWS) + 24
            cv2.putText(board, pos_name(row, col), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA)
    if selected_pos:
        row, col = parse_pos(selected_pos)
        x1 = int(col * width / GRID_COLS)
        y1 = int(row * height / GRID_ROWS)
        x2 = int((col + 1) * width / GRID_COLS) - 1
        y2 = int((row + 1) * height / GRID_ROWS) - 1
        cv2.rectangle(board, (x1, y1), (x2, y2), (0, 255, 255), 5)
    rotation = globals().get('BOARD_CAMERA_ROTATION_DEGREES')
    rotation_text = f' | camera rotation {rotation:.1f} deg' if rotation is not None else ''
    cv2.putText(board, f'Auto-oriented: A1 top-left{rotation_text}', (10, height - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2, cv2.LINE_AA)
    return board


def grid_cell_quad(row, col, frame_shape):
    to_frame = frame_transform(frame_shape)
    grid_points = np.float32([[[col, row], [col + 1, row], [col + 1, row + 1], [col, row + 1]]])
    return cv2.perspectiveTransform(grid_points, to_frame).reshape(4, 2).astype(np.float32)


def crop_grid_cell(frame, row, col):
    """Returns (crop, src_quad, dst_quad) -- the quads let a caller map a
    detection made in crop-space back to real frame coordinates (see
    crop_box_to_frame_box/crop_box_center_to_frame/crop_points_to_frame
    below), needed by the grid-crop and per-cell-ArUco detection paths."""
    width, height = GRID_CROP_SIZE
    src = grid_cell_quad(row, col, frame.shape)
    dst = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    transform = cv2.getPerspectiveTransform(src, dst)
    crop = cv2.warpPerspective(frame, transform, (width, height))
    return crop, src, dst


def box_center(box):
    y_min, x_min, y_max, x_max = map(float, box)
    return (x_min + x_max) / 2.0, (y_min + y_max) / 2.0


def grid_position_for_point(point, frame_shape):
    point_arr = np.float32([[[float(point[0]), float(point[1])]]])
    grid_x, grid_y = cv2.perspectiveTransform(point_arr, board_transform(frame_shape))[0][0]
    col = int(np.clip(np.floor(grid_x), 0, GRID_COLS - 1))
    row = int(np.clip(np.floor(grid_y), 0, GRID_ROWS - 1))
    return {
        'row': row,
        'col': col,
        'cell': pos_name(row, col),
        'center': (float(point[0]), float(point[1])),
    }


def grid_position_for_aruco_id(marker_id, fallback_center):
    marker_id = int(marker_id)
    cell_index = marker_id % (GRID_ROWS * GRID_COLS)
    row = cell_index // GRID_COLS
    col = cell_index % GRID_COLS
    return {
        'row': row,
        'col': col,
        'cell': pos_name(row, col),
        'center': tuple(map(float, fallback_center)),
    }


def aruco_grid_position(marker_id, center, frame_shape):
    if CARD_POSITION_MODE == 'aruco_id':
        return grid_position_for_aruco_id(marker_id, center)
    return grid_position_for_point(center, frame_shape)


def crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame_shape):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_corners = np.float32([[
        [x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max],
    ]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_corners = cv2.perspectiveTransform(crop_corners, inverse_transform).reshape(4, 2)
    frame_height, frame_width = frame_shape[:2]
    xs = np.clip(frame_corners[:, 0], 0, frame_width - 1)
    ys = np.clip(frame_corners[:, 1], 0, frame_height - 1)
    return [float(np.min(ys)), float(np.min(xs)), float(np.max(ys)), float(np.max(xs))]


def crop_box_center_to_frame(crop_box, src_quad, dst_quad):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_center = np.float32([[[((x_min + x_max) / 2.0), ((y_min + y_max) / 2.0)]]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    center = cv2.perspectiveTransform(crop_center, inverse_transform)[0][0]
    return (float(center[0]), float(center[1]))


def crop_points_to_frame(points, src_quad, dst_quad):
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_points = cv2.perspectiveTransform(np.float32([points]), inverse_transform)[0]
    return frame_points.astype(np.float32)


def draw_grid(frame):
    frame = frame.copy()
    to_frame = frame_transform(frame.shape)
    for col in range(GRID_COLS + 1):
        p1, p2 = cv2.perspectiveTransform(np.float32([[[col, 0]], [[col, GRID_ROWS]]]), to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)
    for row in range(GRID_ROWS + 1):
        p1, p2 = cv2.perspectiveTransform(np.float32([[[0, row]], [[GRID_COLS, row]]]), to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)
    return frame


def draw_label(frame, label, x, y, color=(255, 255, 255)):
    size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    y = max(y, size[1] + 8)
    cv2.rectangle(frame, (x - size[0] // 2 - 4, y - size[1] - 8), (x + size[0] // 2 + 4, y + 4), (0, 0, 0), -1)
    return cv2.putText(frame, label, (x - size[0] // 2, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

### Check yourself

1. Why does labeling the corners by their *job* let the camera be upside down?
2. What does `crop_grid_cell` give YOLO that a full picture wouldn't?
